# Chicago Crime - Top-10 Prediction (Version 2)

**New approach**: Instead of predicting a single beat or crime type, we predict the **top 10** most likely beats and crime types. A prediction is correct if the actual outcome is in our top 10.

- **Beat prediction**: Time-focused — which 10 beats are most likely to have crime at this time? (Police deploy to these 10.)
- **Crime type prediction**: Which 10 crime types are most likely?
- **Evaluation**: Recall@10 (hit rate = % of samples where actual was in predicted top 10)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/processed/crimes_cleaned.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (2441506, 28)


,ID,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,...,DayOfWeek,DayOfWeekName,Hour,Quarter,WeekOfYear,IsWeekend,Lat_bin,Lon_bin,HourBin,Crime_Category
0,14135339,2026-03-13 00:00:00,075XX S KINGSTON AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,0,0,421,...,4,Friday,0,1,11,False,2087,-4379,0,Theft
1,14135179,2026-03-13 00:00:00,050XX N MARINE DR,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE - GARAGE,0,0,2024,...,4,Friday,0,1,11,False,2098,-4383,0,Property
2,14138214,2026-03-13 00:00:00,075XX S STONY ISLAND AVE,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,HOSPITAL BUILDING / GROUNDS,0,0,411,...,4,Friday,0,1,11,False,2087,-4380,0,Violent


## 1. Ensure Temporal Features & Sample

If Date is not parsed, add temporal features. Sample for faster experimentation.

In [2]:
if 'Hour' not in df.columns and 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Month'] = df['Date'].dt.month
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['Hour'] = df['Date'].dt.hour
    df['Quarter'] = df['Date'].dt.quarter
    df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
    df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

if df['IsWeekend'].dtype == bool:
    df['IsWeekend'] = df['IsWeekend'].astype(int)

SAMPLE_SIZE = 200000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42)
print("Sample size:", len(df))
print("Columns:", [c for c in df.columns if c in ['Hour', 'DayOfWeek', 'Month', 'Beat', 'Crime_Category', 'Primary Type', 'HourBin']])

Sample size: 200000
Columns: ['Primary Type', 'Beat', 'Month', 'DayOfWeek', 'Hour', 'HourBin', 'Crime_Category']


## 2. Top-K Evaluation

Recall@K: fraction of samples where the actual class is in the predicted top-K.

In [3]:
def recall_at_k(y_true, y_pred_proba, classes, k=10):
    """
    y_true: actual labels (same encoding as classes)
    y_pred_proba: (n_samples, n_classes) probability matrix
    classes: list of class labels in same order as proba columns
    Returns: fraction of samples where actual is in top-k predicted
    """
    hits = 0
    for i, true_label in enumerate(y_true):
        top_k_indices = np.argsort(y_pred_proba[i])[::-1][:k]
        top_k_labels = [classes[j] for j in top_k_indices]
        if true_label in top_k_labels:
            hits += 1
    return hits / len(y_true) if len(y_true) > 0 else 0.0

def run_top10_model(X_tr, X_te, y_tr, y_te, model, task_name, k=10):
    """Train model, predict top-K, return Recall@K."""
    le = LabelEncoder()
    y_tr_enc = le.fit_transform(y_tr)
    y_te_enc = le.transform(y_te)
    classes = list(le.classes_)
    
    model.fit(X_tr, y_tr_enc)
    proba = model.predict_proba(X_te)
    recall = recall_at_k(y_te_enc, proba, list(range(len(classes))), k=k)
    print(f"{task_name}: Recall@{k} = {recall:.4f}")
    return recall, le

## 3. Historical Baseline (Time-Only)

For beat prediction: at each time slot (Hour, DayOfWeek, Month), the top 10 beats are the 10 most common beats in that slot historically. No ML — just aggregation.

In [4]:
def historical_top_k_beats(df_train, df_test, k=10):
    """For each test row, return top-K beats by (Hour, DayOfWeek, Month) from train."""
    agg = df_train.groupby(['Hour', 'DayOfWeek', 'Month'])['Beat'].apply(
        lambda x: x.value_counts().head(k).index.tolist()
    ).to_dict()
    
    hits = 0
    for _, row in df_test.iterrows():
        key = (row['Hour'], row['DayOfWeek'], row['Month'])
        top_k = agg.get(key, df_train['Beat'].value_counts().head(k).index.tolist())
        if row['Beat'] in top_k:
            hits += 1
    return hits / len(df_test) if len(df_test) > 0 else 0.0

def historical_top_k_crimes(df_train, df_test, k=10, target_col='Crime_Category'):
    """For each test row, return top-K crime types by (Hour, DayOfWeek, Month)."""
    agg = df_train.groupby(['Hour', 'DayOfWeek', 'Month'])[target_col].apply(
        lambda x: x.value_counts().head(k).index.tolist()
    ).to_dict()
    
    hits = 0
    for _, row in df_test.iterrows():
        key = (row['Hour'], row['DayOfWeek'], row['Month'])
        top_k = agg.get(key, df_train[target_col].value_counts().head(k).index.tolist())
        if row[target_col] in top_k:
            hits += 1
    return hits / len(df_test) if len(df_test) > 0 else 0.0

## 4. Task 1 — Top-10 Beat Prediction (Time-Focused)

Use **only temporal features** — hour, day, month, etc. The hypothesis: crime location depends heavily on time.

In [5]:
feat_beat_time = ['Hour', 'DayOfWeek', 'Month', 'Quarter', 'IsWeekend', 'WeekOfYear']

# Ensure numeric; keep top 150 beats to avoid too many classes (stratify fails)
top_beats = df['Beat'].value_counts().head(150).index.tolist()
df_beat = df[df['Beat'].isin(top_beats)][feat_beat_time + ['Beat']].copy()
for c in feat_beat_time:
    df_beat[c] = pd.to_numeric(df_beat[c], errors='coerce')
df_beat = df_beat.dropna()

X_b = df_beat[feat_beat_time].astype(float)
y_b = df_beat['Beat'].astype(str)

# Random split (stratify with 150 classes can fail)
train_idx, test_idx = train_test_split(df_beat.index, test_size=0.2, random_state=42)
X_b_tr = X_b.loc[train_idx]
X_b_te = X_b.loc[test_idx]
y_b_tr = y_b.loc[train_idx]
y_b_te = y_b.loc[test_idx]
df_beat_tr = df_beat.loc[train_idx]
df_beat_te = df_beat.loc[test_idx]

print("=== Beat: Historical baseline (time-only, no ML) ===")
hist_beat = historical_top_k_beats(df_beat_tr, df_beat_te, k=10)
print(f"Historical Recall@10: {hist_beat:.4f}")

print("\n=== Beat: ML models (time features, predict_proba -> top 10) ===")
scaler_b = StandardScaler()
X_b_tr_s = scaler_b.fit_transform(X_b_tr)
X_b_te_s = scaler_b.transform(X_b_te)

for name, mdl in [
    ('LR', LogisticRegression(max_iter=500, random_state=42)),
    ('RF', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('GB', GradientBoostingClassifier(n_estimators=100, random_state=42)),
]:
    r, _ = run_top10_model(X_b_tr_s, X_b_te_s, y_b_tr, y_b_te, mdl, f'Beat_{name}', k=10)

=== Beat: Historical baseline (time-only, no ML) ===
Historical Recall@10: 0.0810

=== Beat: ML models (time features, predict_proba -> top 10) ===
Beat_LR: Recall@10 = 0.1122
Beat_RF: Recall@10 = 0.0758
Beat_GB: Recall@10 = 0.1114


## 5. Task 2 — Top-10 Crime Type Prediction

Predict top 10 crime categories. Can use temporal + optional location features.

In [6]:
# Use Crime_Category (5 groups) or Primary Type (more granular)
target_crime = 'Crime_Category'  # 5 classes: Violent, Theft, Property, Narcotics, Other

feat_crime = ['Hour', 'DayOfWeek', 'Month', 'Quarter', 'IsWeekend']
if 'Beat' in df.columns:
    feat_crime = ['Beat'] + feat_crime  # beat gives location context

df_c = df[feat_crime + [target_crime]].dropna()
X_c = df_c[feat_crime].astype(float)
y_c = df_c[target_crime]

X_c_tr, X_c_te, y_c_tr, y_c_te = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

train_idx_c, test_idx_c = train_test_split(df_c.index, test_size=0.2, random_state=42, stratify=y_c)
df_c_tr = df_c.loc[train_idx_c]
df_c_te = df_c.loc[test_idx_c]

print("=== Crime: Historical baseline ===")
n_classes = y_c.nunique()
k_crime = min(10, n_classes)  # if < 10 classes, use all
hist_crime = historical_top_k_crimes(df_c_tr, df_c_te, k=k_crime, target_col=target_crime)
print(f"Historical Recall@{k_crime}: {hist_crime:.4f}")

print(f"\n=== Crime: ML models (top {k_crime}) ===")
scaler_c = StandardScaler()
X_c_tr_s = scaler_c.fit_transform(X_c_tr)
X_c_te_s = scaler_c.transform(X_c_te)

for name, mdl in [
    ('LR', LogisticRegression(max_iter=500, random_state=42)),
    ('RF', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('GB', GradientBoostingClassifier(n_estimators=100, random_state=42)),
]:
    r, _ = run_top10_model(X_c_tr_s, X_c_te_s, y_c_tr, y_c_te, mdl, f'Crime_{name}', k=k_crime)

=== Crime: Historical baseline ===
Historical Recall@5: 0.9980

=== Crime: ML models (top 5) ===
Crime_LR: Recall@5 = 1.0000
Crime_RF: Recall@5 = 1.0000
Crime_GB: Recall@5 = 1.0000


## 6. Summary

- **Beat**: Time-focused features; Recall@10 = "did we include the actual beat in our top 10?"
- **Crime**: Similar; with 5 categories, Recall@5 is max. Use Primary Type for top-10 if desired.
- **Historical baseline**: Strong for time-driven patterns; ML can add value if location/other features help.

In [7]:
print("Done. Top-10 approach is more realistic for deployment:")
print("  - Police deploy to 10 beats, not one.")
print("  - A hit = actual beat was in our predicted top 10.")

Done. Top-10 approach is more realistic for deployment:
  - Police deploy to 10 beats, not one.
  - A hit = actual beat was in our predicted top 10.
